# 05 - Combine All Datasets: Demographics + Weather + Livestock + Smoking

This notebook merges four data sources into a single dataset for modeling:

1. Demographics (ACS socioeconomic variables + lung cancer mortality, from notebook 04)
2. Weather (CAMS/ERA5 atmospheric and meteorological variables)
3. Livestock (FAO GLW livestock density by species)
4. Smoking (CHR adult smoking rate + is_post_2015 dummy, **new for this paper**)

All merges are inner joins on `['fips', 'Year']`.

In [1]:
import pandas as pd
import os

## 1. Load Demographics Dataset (from Notebook 04)

In [2]:
demographics_df = pd.read_csv('../data/demographics_final/combined_all_years_cleaned_final.csv',
                              dtype={'State_FIPS': str, 'County_FIPS': str})

# Create standardized 5-digit FIPS column
demographics_df['fips'] = demographics_df['State_FIPS'].str.zfill(2) + demographics_df['County_FIPS'].str.zfill(3)

# Drop duplicate lowercase year column
if 'year' in demographics_df.columns and 'Year' in demographics_df.columns:
    demographics_df = demographics_df.drop(columns=['year'])

print(f"Demographics shape: {demographics_df.shape}")
print(f"Years: {sorted(demographics_df['Year'].unique())}")
print(f"FIPS sample: {demographics_df['fips'].head().tolist()}")
demographics_df.head()

Demographics shape: (24871, 23)
Years: [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019]
FIPS sample: ['01001', '01003', '01005', '01007', '01009']


,County,State,Lung Cancer Mortality Rate,State_FIPS,County_FIPS,Fips,Poverty Rate,High School Degree or Higher (%),Bachelor's Degree or Higher (%),Median Age,...,Median Household Income,Unemployment Rate,Year,White Population (%),Hispanic Population (%),Black Population (%),Households with No Vehicle (%),Rent Burden (+50% of HI),Single Mother Families (%),fips
0,Autauga County,Alabama,0.000756,1,1,1001,11.6,85.0,24.4,37.0,...,53773.0,8.6,2012,78.922880,2.399707,18.098553,5.136952,20.377868,10.955253,01001
1,Baldwin County,Alabama,0.000620,1,3,1003,13.3,87.0,29.3,41.2,...,50706.0,8.5,2012,86.373659,4.319802,9.286892,3.081745,18.386173,8.876182,01003
2,Barbour County,Alabama,0.000790,1,5,1005,26.1,70.2,13.0,38.2,...,31889.0,13.5,2012,48.957006,4.969238,46.033711,9.975592,25.047801,16.343447,01005
3,Bibb County,Alabama,0.000827,1,7,1007,16.5,71.5,8.2,39.4,...,36824.0,10.5,2012,76.678818,1.840221,21.753261,5.117790,16.062544,9.766764,01007
4,Blount County,Alabama,0.000754,1,9,1009,14.7,73.9,12.0,39.1,...,45192.0,10.0,2012,94.850868,8.084781,1.312080,3.832438,16.381480,6.856960,01009


## 2. Load Weather Dataset

In [3]:
weather_df = pd.read_pickle('../data/weather/final_dataset_103_features.pkl')

# Standardize FIPS and Year columns
weather_df['fips'] = weather_df['fips'].astype(str).str.zfill(5)
weather_df = weather_df.rename(columns={'year': 'Year'})

# Drop artifact column from life expectancy pipeline
if 'MeanLifeExpectency' in weather_df.columns:
    weather_df = weather_df.drop(columns=['MeanLifeExpectency'])

print(f"Weather shape: {weather_df.shape}")
print(f"Years: {sorted(weather_df['Year'].unique())}")
print(f"FIPS sample: {weather_df['fips'].head().tolist()}")

Weather shape: (52320, 105)
Years: [2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019]
FIPS sample: ['01001', '01003', '01005', '01007', '01009']


## 3. Load Livestock Dataset

In [4]:
livestock_dfs = []
for year in range(2012, 2020):
    df = pd.read_csv(f'../data/livestock/county_mean_{year}.csv')
    livestock_dfs.append(df)

livestock_df = pd.concat(livestock_dfs, ignore_index=True)

# Create standardized FIPS column and rename year
livestock_df['fips'] = livestock_df['STATEFP'].astype(str).str.zfill(2) + livestock_df['COUNTYFP'].astype(str).str.zfill(3)
livestock_df = livestock_df.rename(columns={'year': 'Year'})
livestock_df = livestock_df.drop(columns=['STATEFP', 'COUNTYFP', 'NAME', 'GEOID'])

print(f"Livestock shape: {livestock_df.shape}")
print(f"Years: {sorted(livestock_df['Year'].unique())}")
print(f"Columns: {livestock_df.columns.tolist()}")

Livestock shape: (25576, 10)
Years: [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019]
Columns: ['Year', 'Buffalo', 'Cattle', 'Chicken', 'Duck', 'Goat', 'Horse', 'Pig', 'Sheep', 'fips']


## 4. Load Smoking Dataset

In [5]:
smoking_df = pd.read_csv('../data/processed/smoking_by_year/smoking_all_years.csv')
smoking_df['fips'] = smoking_df['fips'].astype(str).str.zfill(5)

print(f"Smoking shape: {smoking_df.shape}")
print(f"Years: {sorted(smoking_df['Year'].unique())}")
print(f"Columns: {smoking_df.columns.tolist()}")

Smoking shape: (23023, 4)
Years: [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019]
Columns: ['fips', 'Year', 'smoking_rate', 'is_post_2015']


## 5. Merge All Datasets

In [6]:
# Step 1: Demographics + Weather
merged = pd.merge(demographics_df, weather_df, on=['fips', 'Year'], how='inner')
print(f"After demographics + weather merge: {merged.shape}")

# Step 2: + Livestock
merged = pd.merge(merged, livestock_df, on=['fips', 'Year'], how='inner')
print(f"After livestock merge: {merged.shape}")

# Step 3: + Smoking (NEW for lung cancer paper)
merged = pd.merge(merged, smoking_df, on=['fips', 'Year'], how='inner')
print(f"After smoking merge: {merged.shape}")

After demographics + weather merge: (24604, 126)
After livestock merge: (24487, 134)
After smoking merge: (22540, 136)


## 6. Clean Up and Final Checks

In [7]:
# Drop redundant columns
cols_to_drop = [c for c in ['State_FIPS', 'County_FIPS', 'Fips', 'year'] if c in merged.columns]
# Also drop any _x/_y suffix duplicates
cols_to_drop += [c for c in merged.columns if c.endswith('_x') or c.endswith('_y')]
if cols_to_drop:
    merged = merged.drop(columns=cols_to_drop)
    print(f"Dropped columns: {cols_to_drop}")

print(f"\nFinal dataset shape: {merged.shape}")
print(f"\nObservations per year:")
print(merged['Year'].value_counts().sort_index())
print(f"\nMissing values (non-zero only):")
missing = merged.isnull().sum()
missing_nonzero = missing[missing > 0]
if len(missing_nonzero) == 0:
    print("  None")
else:
    print(missing_nonzero)

print(f"\nAll columns ({len(merged.columns)}):")
print(merged.columns.tolist())

Dropped columns: ['State_FIPS', 'County_FIPS', 'Fips']

Final dataset shape: (22540, 133)

Observations per year:
Year
2012    2465
2013    2486
2014    2673
2015    2673
2016    3061
2017    3061
2018    3060
2019    3061
Name: count, dtype: int64

Missing values (non-zero only):
  None

All columns (133):
['County', 'State', 'Lung Cancer Mortality Rate', 'Poverty Rate', 'High School Degree or Higher (%)', "Bachelor's Degree or Higher (%)", 'Median Age', 'Gini Index', 'Disability Rate', 'Total Population', 'Median Household Income', 'Unemployment Rate', 'Year', 'White Population (%)', 'Hispanic Population (%)', 'Black Population (%)', 'Households with No Vehicle (%)', 'Rent Burden (+50% of HI)', 'Single Mother Families (%)', 'fips', '2m dew point temperature', '2m temperature', 'Black carbon AOD at 550 nm', 'Dust AOD at 550 nm', 'Land-sea mask', 'Mean sea level pressure', 'Organic matter AOD at 550 nm', 'PM$_1$', 'PM$_{2.5}$', 'PM$_{10}$', 'Sea salt AOD at 550 nm', 'Sulphate AOD at 55

## 7. Save Final Combined Dataset

In [8]:
output_dir = '../data/combined_final/'
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, 'final_combined_all_variables.csv')
merged.to_csv(output_path, index=False)
print(f"Saved {len(merged)} rows to {output_path}")

Saved 22540 rows to ../data/combined_final/final_combined_all_variables.csv


## 8. Verify Output File

In [9]:
df_check = pd.read_csv(output_path)
print(f"Output file: {output_path}")
print(f"Shape: {df_check.shape}")
print(f"Columns: {df_check.columns.tolist()}")
print(f"\nSmoking rate present: {'smoking_rate' in df_check.columns}")
print(f"is_post_2015 present: {'is_post_2015' in df_check.columns}")
df_check.head()

Output file: ../data/combined_final/final_combined_all_variables.csv
Shape: (22540, 133)
Columns: ['County', 'State', 'Lung Cancer Mortality Rate', 'Poverty Rate', 'High School Degree or Higher (%)', "Bachelor's Degree or Higher (%)", 'Median Age', 'Gini Index', 'Disability Rate', 'Total Population', 'Median Household Income', 'Unemployment Rate', 'Year', 'White Population (%)', 'Hispanic Population (%)', 'Black Population (%)', 'Households with No Vehicle (%)', 'Rent Burden (+50% of HI)', 'Single Mother Families (%)', 'fips', '2m dew point temperature', '2m temperature', 'Black carbon AOD at 550 nm', 'Dust AOD at 550 nm', 'Land-sea mask', 'Mean sea level pressure', 'Organic matter AOD at 550 nm', 'PM$_1$', 'PM$_{2.5}$', 'PM$_{10}$', 'Sea salt AOD at 550 nm', 'Sulphate AOD at 550 nm', 'Surface geopotential', 'Surface pressure', 'Total AOD at 469 nm', 'Total AOD at 550 nm', 'Total AOD at 670 nm', 'Total AOD at 865 nm', 'Total AOD at 1240 nm', 'Total column carbon monoxide', 'Total colum

,County,State,Lung Cancer Mortality Rate,Poverty Rate,High School Degree or Higher (%),Bachelor's Degree or Higher (%),Median Age,Gini Index,Disability Rate,Total Population,...,Buffalo,Cattle,Chicken,Duck,Goat,Horse,Pig,Sheep,smoking_rate,is_post_2015
0,Autauga County,Alabama,0.000756,11.6,85.0,24.4,37.0,0.4122,15.0,54590.0,...,104.304193,765.477347,1460.733157,6.200551,47.556840,54.780249,3.199114,13.548855,0.246,0
1,Baldwin County,Alabama,0.000620,13.3,87.0,29.3,41.2,0.4364,14.0,183226.0,...,0.000000,356.486268,20.764893,2.339871,20.358615,74.033112,3.038270,4.505194,0.227,0
2,Barbour County,Alabama,0.000790,26.1,70.2,13.0,38.2,0.4758,20.5,27469.0,...,74.777062,595.600612,150608.306932,6.572719,11.807808,24.807758,8.248660,6.213882,0.234,0
3,Bibb County,Alabama,0.000827,16.5,71.5,8.2,39.4,0.4212,16.2,22769.0,...,0.000000,288.782290,516.739571,3.657299,15.416166,42.510005,20.036976,2.450206,0.351,0
4,Blount County,Alabama,0.000754,14.7,73.9,12.0,39.1,0.4112,17.3,57466.0,...,0.000000,1692.635794,549287.154544,16.943384,42.096943,228.072444,9.920152,22.714804,0.227,0
